First of all, import all the necessary modules:

In [ ]:
import optuna
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from numba import njit

Define the backtest functions:

In [14]:
# ====================== JITTED BACKTEST CORE ======================
@njit(fastmath=True, cache=True)
def simulate(closes: np.ndarray, entry_signals: np.ndarray, opposite_cross: np.ndarray,
             rsi_high_hit: np.ndarray, rsi_low_hit: np.ndarray,
             sl_rate: float, tp_rate: float, max_hold: int,
             initial_capital: float, fee_rate: float):
    n = len(closes)
    equity = np.empty(n, dtype=np.float64)
    cash = initial_capital
    position = 0.0
    entry_price = 0.0
    planned_exit = -1
    total_traded = 0.0
    max_lev = 0.0
    trades = 0

    for i in range(n):
        close_p = closes[i]

        # === EXIT FIRST ===
        if position > 0.0:
            hit_opposite = opposite_cross[i]
            hit_r_high = rsi_high_hit[i]
            hit_r_low = rsi_low_hit[i]
            hit_sl = close_p <= entry_price * (1.0 - sl_rate)
            hit_tp = close_p >= entry_price * (1.0 + tp_rate)
            hit_time = (i >= planned_exit)

            if hit_opposite or hit_r_high or hit_r_low or hit_sl or hit_tp or hit_time:
                sell_notional = position * close_p
                total_traded += sell_notional
                cash += sell_notional * (1.0 - fee_rate) if fee_rate > 0.0 else sell_notional
                position = 0.0
                entry_price = 0.0
                planned_exit = -1

        # === ENTRY ===
        if (position == 0.0 and entry_signals[i] and cash > 0.01 and (i + max_hold < n)):
            notional = cash / (1.0 + fee_rate) if fee_rate > 0.0 else cash
            size = notional / close_p
            position = size
            cash = 0.0
            total_traded += notional
            entry_price = close_p
            planned_exit = i + max_hold
            trades += 1

        equity[i] = cash + position * close_p

        if equity[i] > 0.0 and position > 0.0:
            lev = (position * close_p) / equity[i]
            if lev > max_lev:
                max_lev = lev

    return equity, total_traded, max_lev, trades


def compute_metrics(equity: np.ndarray, initial_capital: float, total_traded: float, n: int):
    final_pnl = equity[-1] - initial_capital
    if n < 2:
        return {'final_pnl': final_pnl, 'sharpe': 0.0, 'ann_sharpe': 0.0,
                'ann_turnover': 0.0, 'max_dd_pct': 0.0}

    returns = np.diff(equity) / equity[:-1]
    sharpe = 0.0
    ann_sharpe = 0.0
    if len(returns) > 30 and np.std(returns) > 1e-8:
        sharpe = np.mean(returns) / np.std(returns)
        annual_factor = 365.25 * 24
        ann_sharpe = sharpe * np.sqrt(annual_factor)

    cummax = np.maximum.accumulate(equity)
    dd = (equity - cummax) / cummax
    max_dd_pct = -dd.min() * 100 if dd.min() < 0 else 0.0

    years = n / annual_factor
    ann_turnover = (total_traded / initial_capital) / years if years > 0 else 0.0

    return {
        'final_pnl': final_pnl,
        'sharpe': sharpe,
        'ann_sharpe': ann_sharpe,
        'ann_turnover': ann_turnover,
        'max_dd_pct': max_dd_pct
    }


# ====================== SIGNAL COMPUTATION (all parameters) ======================
def compute_entry_exit_signals(closes, volume, ema_fast, ema_slow, rsi_period,
                               rsi_entry_thresh, vol_sma_period, vol_mult,
                               rsi_high, rsi_low):
    close_s = pd.Series(closes)
    vol_s = pd.Series(volume)

    # EMAs
    ema_f = close_s.ewm(span=ema_fast, adjust=False).mean()
    ema_s = close_s.ewm(span=ema_slow, adjust=False).mean()

    # Crosses
    cross_up = (ema_f.shift(1) <= ema_s.shift(1)) & (ema_f > ema_s)
    opposite_cross = (ema_f.shift(1) >= ema_s.shift(1)) & (ema_f < ema_s)

    # RSI (Wilder)
    delta = close_s.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=rsi_period - 1, adjust=False).mean()
    avg_loss = loss.ewm(com=rsi_period - 1, adjust=False).mean()
    rs = avg_gain / avg_loss
    rsi_val = 100 - (100 / (1 + rs))

    # Volume filter
    vol_sma = vol_s.rolling(window=vol_sma_period, min_periods=vol_sma_period).mean()
    vol_condition = vol_s > vol_mult * vol_sma

    # Entry (cross + RSI + volume) — covers breakout / pullback momentum
    entry_signals = cross_up & (rsi_val > rsi_entry_thresh) & vol_condition & vol_sma.notna()

    # Exit indicator hits
    rsi_high_hit = rsi_val >= rsi_high
    rsi_low_hit = rsi_val <= rsi_low

    return (entry_signals.values.astype(bool),
            opposite_cross.values.astype(bool),
            rsi_high_hit.values.astype(bool),
            rsi_low_hit.values.astype(bool))

Then, run the backtest (specifically for $(x, y) = (10, 3)$):

In [ ]:
csv_path = "data/BTCUSDT-1h-2023-01-01_2026-02-28.csv"
ema_fast, ema_slow, rsi_period, rsi_entry_thresh, vol_sma_period, vol_mult, rsi_high, rsi_low = 11, 24, 25, 75.0, 25, 2.4, 84.6, 22.3
sl, tp, max_hold = 0.0106, 0.0358, 113

print("Loading CSV...")
df = pd.read_csv(csv_path)
df['datetime'] = pd.to_datetime(df["OpenTimestamp[ms]"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)
n = len(df)
print(f"Data loaded: {n:,} minutes (~{n/(60*24):.1f} days)")

closes = df['Close'].values
volume = df['Volume'].values
initial_capital = 1_000_000.0

# ====================== DEFAULT BACKTEST ======================
print(f"\nRunning default (EMA {ema_fast}/{ema_slow}, RSI{rsi_period}>{rsi_entry_thresh}, Vol>{vol_mult}×{vol_sma_period}SMA, SL{sl}%, TP{tp}%, max{max_hold}min)...")
entry_def, opp_def, r_high_def, r_low_def = compute_entry_exit_signals(closes, volume, ema_fast, ema_slow, rsi_period, rsi_entry_thresh, vol_sma_period, vol_mult, rsi_high, rsi_low)

equity_before, traded_before, max_lev_before, trades_before = simulate(closes, entry_def, opp_def, r_high_def, r_low_def, sl, tp, max_hold, initial_capital, 0.0)
equity_after, traded_after, max_lev_after, trades_after = simulate(closes, entry_def, opp_def, r_high_def, r_low_def, sl, tp, max_hold, initial_capital, 0.001)

metrics_before = compute_metrics(equity_before, initial_capital, traded_before, n)
metrics_after = compute_metrics(equity_after, initial_capital, traded_after, n)

print("\n" + "=" * 90)
print("DEFAULT RESULTS")
print("=" * 90)
print(f"Final PnL BEFORE    : ${metrics_before['final_pnl']:,.2f}")
print(f"Final PnL AFTER     : ${metrics_after['final_pnl']:,.2f}")
print(f"Ann. Sharpe BEFORE  : {metrics_before['ann_sharpe']:.2f}")
print(f"Ann. Sharpe AFTER   : {metrics_after['ann_sharpe']:.2f}")
print(f"Max DD BEFORE       : {metrics_before['max_dd_pct']:.2f}%")
print(f"Max DD AFTER        : {metrics_after['max_dd_pct']:.2f}%")
print(f"Total trades BEFORE : {trades_before:,}")
print(f"Total trades AFTER  : {trades_after:,}")
print(f"Ann. Turnover       : {metrics_after['ann_turnover']:.2f}x")
print(f"Max Leverage        : {max_lev_after:.2f}x")
print("=" * 90)

# Save equity curve for Flask dashboard
equity_df = pd.DataFrame({'datetime': df['datetime'], 'pnl_before': equity_before - initial_capital, 'pnl_after': equity_after - initial_capital})
equity_df.to_csv('equity_curve.csv', index=False)
print("Saved 'equity_curve.csv' for the Flask dashboard.")

# Cumulative PnL plot (static)
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before - initial_capital, label='Before 0.1% cost', color='royalblue')
plt.plot(df['datetime'], equity_after - initial_capital, label='After 0.1% cost', color='crimson')
plt.title(f'Mean-Reversion Backtest (x={max_hold}, SL={sl * 100:.2f}, TP={tp * 100:.2f}%)', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Now, we use `optuna` to optimize...

In [ ]:
# ====================== OPTUNA OBJECTIVE (parameters) ======================
def objective(trial, closes, volume, initial_capital, n, fee_rate=0.001):
    ema_fast = trial.suggest_int('ema_fast', 5, 25)
    ema_slow = trial.suggest_int('ema_slow', 15, 60)
    if ema_slow <= ema_fast:
        return -999.0
    rsi_period = trial.suggest_int('rsi_period', 5, 30)
    rsi_entry_thresh = trial.suggest_float('rsi_entry_thresh', 50.0, 75.0)
    vol_sma_period = trial.suggest_int('vol_sma_period', 10, 50)
    vol_mult = trial.suggest_float('vol_mult', 1.5, 5.0)
    rsi_high = trial.suggest_float('rsi_high', 65.0, 85.0)
    rsi_low = trial.suggest_float('rsi_low', 15.0, 35.0)
    max_hold = trial.suggest_int('max_hold', 10, 300)
    sl = trial.suggest_float('sl', 0.001, 0.015)
    tp = trial.suggest_float('tp', 0.002, 0.06)

    entry, opp, r_high, r_low = compute_entry_exit_signals(
        closes, volume, ema_fast, ema_slow, rsi_period,
        rsi_entry_thresh, vol_sma_period, vol_mult, rsi_high, rsi_low)

    equity, traded, _, _ = simulate(closes, entry, opp, r_high, r_low,
                                    sl, tp, max_hold, initial_capital, fee_rate)
    metrics = compute_metrics(equity, initial_capital, traded, n)
    return metrics['ann_sharpe']


# ====================== OPTUNA OPTIMIZATION ======================
n_trials = 2_000

print(f"\nStarting Optuna ({n_trials} trials) — tuning 11 parameters to maximize Sharpe...")
start_time = time.time()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=67))
study.optimize(lambda trial: objective(trial, closes, volume, initial_capital, n, fee_rate=0.001),
               n_trials=n_trials, show_progress_bar=True)

best = study.best_params
best_sharpe = study.best_value
print(f"\nOPTUNA BEST: EMA{best['ema_fast']}/{best['ema_slow']}, RSI({best['rsi_period']})>{best['rsi_entry_thresh']:.1f}, "
      f"Vol>{best['vol_mult']:.1f}×{best['vol_sma_period']}, RSI exit {best['rsi_high']:.1f}/{best['rsi_low']:.1f}, "
      f"max_hold={best['max_hold']}, SL={best['sl']*100:.2f}%, TP={best['tp']*100:.2f}% → Sharpe = {best_sharpe:.4f}")
print(f"Finished in {time.time() - start_time:.1f}s")

# ====================== RUN BEST & SAVE FOR DASHBOARD ======================
print("\nRunning best parameters...")
entry_best, opp_best, r_high_best, r_low_best = compute_entry_exit_signals(closes, volume, best['ema_fast'], best['ema_slow'], best['rsi_period'], best['rsi_entry_thresh'], best['vol_sma_period'], best['vol_mult'], best['rsi_high'], best['rsi_low'])

equity_before_best, _, _, _ = simulate(closes, entry_best, opp_best, r_high_best, r_low_best, best['sl'], best['tp'], best['max_hold'], initial_capital, 0.0)
equity_after_best, traded_best, max_lev_best, trades_best = simulate(closes, entry_best, opp_best, r_high_best, r_low_best, best['sl'], best['tp'], best['max_hold'], initial_capital, 0.001)
metrics_best = compute_metrics(equity_after_best, initial_capital, traded_best, n)

equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl_before': equity_before_best - initial_capital,
    'pnl_after': equity_after_best - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("✅ Saved 'equity_curve.csv' (BEST parameters)")

print("\n" + "="*95)
print(f"BEST RESULTS (EMA{best['ema_fast']}/{best['ema_slow']}, RSI({best['rsi_period']})>{best['rsi_entry_thresh']:.1f}, "
      f"Vol>{best['vol_mult']:.1f}×{best['vol_sma_period']}, x={best['max_hold']}, "
      f"SL={best['sl']*100:.2f}%, TP={best['tp']*100:.2f}%)")
print("="*95)
print(f"Final PnL AFTER  : ${metrics_best['final_pnl']:,.2f}")
print(f"Sharpe AFTER     : {metrics_best['sharpe']:.4f}")
print(f"Ann. Sharpe      : {metrics_best['ann_sharpe']:.2f}")
print(f"Max DD           : {metrics_best['max_dd_pct']:.2f}%")
print(f"Total trades     : {trades_best:,}")
print(f"Ann. Turnover    : {metrics_best['ann_turnover']:.2f}x")
print(f"Max Leverage     : {max_lev_best:.2f}x")
print("="*95)

plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before_best - initial_capital, label='Before cost', color='royalblue')
plt.plot(df['datetime'], equity_after_best - initial_capital, label='After cost', color='crimson')
plt.title(f'BEST Strategy (EMA{best["ema_fast"]}/{best["ema_slow"]}, RSI{best["rsi_period"]}, '
          f'x={best["max_hold"]}, SL={best["sl"]*100:.2f}%, TP={best["tp"]*100:.2f}%)', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()